In [1]:
import pandas as pd
from prophet import Prophet
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import streamlit as st
import toml
import joblib
import plotly.express as px


c:\Users\timwy\Documents\GitHub\streamlit_bee_dasbhoard\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# MongoDB Atlas Connection
secrets = toml.load("../.streamlit//secrets.toml")
uri = secrets["mongodb"]["uri"]
client = MongoClient(uri, server_api=ServerApi('1'))
db = client["beehive_monitoring"]  # Database name
collection = db["bee_sensor_telemetry"]  # Collection name

# Fetch data from the collection
data = pd.DataFrame(list(collection.find()))
print(data.dtypes)
data

_id             object
beehive_id      object
weight         float64
temperature     object
humidity        object
timestamp       object
dtype: object


,_id,beehive_id,weight,temperature,humidity,timestamp
0,679665ae0d8001aa456a10e1,2,24.73,19.9,74.0,2025-01-26 17:41:15
1,679668030d8001aa456a10e2,2,24.74,20.1,75.0,2025-01-26 17:51:15
2,6796697ee0ee352edb1f5f26,2,46.41,20.0,93.0,2020-10-08T20:18:36+02:00
3,6796697ee0ee352edb1f5f29,2,46.40,20.2,93.0,2020-10-08T20:48:49+02:00
4,6796697ee0ee352edb1f5f36,2,46.42,20.4,93.0,2020-10-08T22:59:47+02:00
...,...,...,...,...,...,...
304002,67f03ebf94c22865951dbb1f,1,36.96,30.8,64.0,2025-04-04 22:19:10
304003,67f03fad0f9d50227a655334,2,31.62,None,None,2025-04-04 22:23:08
304004,67f0411894c22865951dc2d2,1,36.95,30.7,64.0,2025-04-04 22:29:11
304005,67f042040f9d50227a655786,2,31.61,None,None,2025-04-04 22:33:07


In [3]:
# Prepare data
data['timestamp'] = pd.to_datetime(data['timestamp'], format='mixed', utc=True)
data = data.dropna(subset=['weight', 'timestamp', 'beehive_id'])
display(data.head())

# Exclude '_id' column from the aggregation
agg_data = data.groupby(['beehive_id', pd.Grouper(key='timestamp', freq='D')])['weight'].mean().reset_index().dropna()

# ValueError: Column ds has timezone specified, which is not supported. Remove timezone.
agg_data['timestamp'] = agg_data['timestamp'].dt.tz_localize(None)
agg_data['weight'] = agg_data['weight'].astype(float)

display(agg_data.head())

,_id,beehive_id,weight,temperature,humidity,timestamp
0,679665ae0d8001aa456a10e1,2,24.73,19.9,74.0,2025-01-26 17:41:15+00:00
1,679668030d8001aa456a10e2,2,24.74,20.1,75.0,2025-01-26 17:51:15+00:00
2,6796697ee0ee352edb1f5f26,2,46.41,20.0,93.0,2020-10-08 18:18:36+00:00
3,6796697ee0ee352edb1f5f29,2,46.40,20.2,93.0,2020-10-08 18:48:49+00:00
4,6796697ee0ee352edb1f5f36,2,46.42,20.4,93.0,2020-10-08 20:59:47+00:00


,beehive_id,timestamp,weight
0,1,2020-10-14,49.670556
1,1,2020-10-15,49.641357
2,1,2020-10-16,49.579226
3,1,2020-10-17,49.483846
4,1,2020-10-18,49.423916


In [4]:
# Train one model per beehive
models = {}
for beehive_id, df_group in agg_data.groupby('beehive_id'):
    training_data = df_group[['timestamp', 'weight']].rename(columns={'timestamp': 'ds', 'weight': 'y'})
    model = Prophet()
    model.fit(training_data)

    # Save model
    joblib.dump(model, f"../models/prophet_model_{beehive_id}.pkl")
    models[beehive_id] = model

22:42:35 - cmdstanpy - INFO - Chain [1] start processing
22:42:36 - cmdstanpy - INFO - Chain [1] done processing
22:42:36 - cmdstanpy - INFO - Chain [1] start processing
22:42:36 - cmdstanpy - INFO - Chain [1] done processing


# Forecasting


In [ ]:
st.title("Beehive Weight Forecast (Monthly, Daily Granularity)")

beehive_ids = "1"


# Load model and data
model = joblib.load(f"../models/prophet_model_{beehive_id}.pkl")
recent_data = agg_data[(agg_data['beehive_id'] == beehive_id) & 
                       (agg_data['timestamp'] >= pd.Timestamp('2025-01-01')) & 
                       (agg_data['timestamp'] <= pd.Timestamp('2025-04-04'))].rename(columns={'timestamp': 'ds', 'weight': 'y'})

# Create future DataFrame for 30 days
future = model.make_future_dataframe(periods=30, freq='D')
forecast = model.predict(future)

# Merge historical + forecast (optional smoothing)
historical = recent_data.copy()

# Plot
fig = px.line(title=f"Beehive {beehive_id}: Weight Forecast (Next 30 Days)")
fig.add_scatter(x=historical['ds'], y=historical['y'], name="Historical")
fig.add_scatter(x=forecast['ds'], y=forecast['yhat'], name="Forecast")
fig.add_scatter(x=forecast['ds'], y=forecast['yhat_lower'], name="Lower Bound", line=dict(dash="dot"))
fig.add_scatter(x=forecast['ds'], y=forecast['yhat_upper'], name="Upper Bound", line=dict(dash="dot"))

fig.show()


2025-04-04 22:42:36.529 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-04 22:42:36.958 
  command:

    streamlit run c:\Users\timwy\Documents\GitHub\streamlit_bee_dasbhoard\.venv\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-04-04 22:42:36.959 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [6]:
display(forecast)

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,weekly,weekly_lower,weekly_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2020-10-08,29.345801,26.374610,39.113581,29.345801,29.345801,3.355895,3.355895,3.355895,-0.042919,-0.042919,-0.042919,3.398813,3.398813,3.398813,0.0,0.0,0.0,32.701695
1,2020-10-09,29.342852,26.495714,38.789168,29.342852,29.342852,3.443739,3.443739,3.443739,0.268379,0.268379,0.268379,3.175360,3.175360,3.175360,0.0,0.0,0.0,32.786591
2,2020-10-10,29.339904,26.637622,38.649738,29.339904,29.339904,3.211049,3.211049,3.211049,0.235411,0.235411,0.235411,2.975639,2.975639,2.975639,0.0,0.0,0.0,32.550953
3,2020-10-11,29.336955,25.958145,38.440522,29.336955,29.336955,2.920828,2.920828,2.920828,0.121464,0.121464,0.121464,2.799364,2.799364,2.799364,0.0,0.0,0.0,32.257783
4,2020-10-12,29.334007,25.750768,38.878945,29.334007,29.334007,2.585321,2.585321,2.585321,-0.060477,-0.060477,-0.060477,2.645798,2.645798,2.645798,0.0,0.0,0.0,31.919328
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
867,2025-04-30,26.951669,27.071671,39.456666,26.934545,26.972121,6.464967,6.464967,6.464967,-0.212045,-0.212045,-0.212045,6.677012,6.677012,6.677012,0.0,0.0,0.0,33.416636
868,2025-05-01,26.946232,28.559631,41.264021,26.926059,26.969005,7.731163,7.731163,7.731163,-0.042919,-0.042919,-0.042919,7.774082,7.774082,7.774082,0.0,0.0,0.0,34.677395
869,2025-05-02,26.940795,29.801010,42.153261,26.918726,26.964970,9.099953,9.099953,9.099953,0.268379,0.268379,0.268379,8.831574,8.831574,8.831574,0.0,0.0,0.0,36.040748
870,2025-05-03,26.935358,30.809163,43.374205,26.910268,26.961019,10.066937,10.066937,10.066937,0.235411,0.235411,0.235411,9.831527,9.831527,9.831527,0.0,0.0,0.0,37.002295
